# <center>Big Data &ndash; Exercise 1</center>
## <center>Fall 2026 &ndash; Week 1 &ndash; ETH Zurich</center>

### Aims
- **After this exercise:** 
    - Understand the SQL language and its common query patterns.
    - Understand the 'table' data shape, normalization, and when they can (and should) be used.
    - be able to query data in tables with the SQL language.
- **Later in the semester:** 
    - Relate these language features and query patterns relative to other data shapes, technologies, and the languages designed to query them.
    - Understand when tables are not the appropriate shape for your data and when you can (and should) throw normalization away!

### Prerequisites
In this exercise, you will brush-up the fundamental concepts of relational databases and SQL. If you haven't taken an introductory databases course (or want to refresh your knowledge) we recommend you to read the following:

Garcia-Molina, Ullman, Widom: Database Systems: The Complete Book. Pearson, 2. Edition, 2008. (Chapters 1, 2, 3, and 6) [Available in the ETH Library] [[Online]](https://ebookcentral.proquest.com/lib/ethz/detail.action?pq-origsite=primo&docID=5832965) [[Selected solutions]](http://infolab.stanford.edu/~ullman/dscbsols/sols.html).

Or have a look at the recordings from Information Systems for Engineers - ETH Zurich, available on [[YouTube]](https://www.youtube.com/c/GhislainFournysLectures).

### Database Set-up
We will be once again working in the ExamMagicBox (you can find it in the following [[link]](https://polybox.ethz.ch/index.php/s/BXf5ndnrk6AiBJe) if you have not downloaded it yet): please drag this Notebook in the folder. Just like last week, activate the docker container for the exercise sheet with `docker compose up`; please wait for the message `PostgreSQL init process complete; ready for start up` in the docker logs before proceeding! Alternatively you can start the Docker with `docker compose up -d` and wait for the command to execute: please note that you are creating the containers in the background this way. You can then type `docker compose down` when you are done.

As before, we set up our connection to the database and enable use of `%sql` and `%%sql`.

In [1]:
%reload_ext sql
%sql postgresql://postgres:example@db

Connecting to 'postgresql://postgres:***@db'

In [2]:
%%sql
SELECT version();

Running query in 'postgresql://postgres:***@db'

1 rows affected.

version
"PostgreSQL 17.6 (Debian 17.6-2.pgdg13+1) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit"


In [3]:
# Set this to the number you prefer for output rows
%config SqlMagic.displaylimit = 25

### Origin of the data
You can find more information on the dataset in the following links
- [Discogs](https://www.discogs.com/)
- [Discogs XML data dumps](http://data.discogs.com/)

If you do not want to use Docker or it does not work you can download the dataset from this [link](https://cloud.inf.ethz.ch/s/DtjCHTLRHT39BRN/download/discogs.dump.xz), see `postgres-init.sh` to see how to import it)

## Exercise 1: Explore the dataset
We want to first understand the dataset a bit better. You will find some queries below to help you explore the schema.

### List tables
The following query retrieves a list of tables in the database from a system table describing the current database.

In [4]:
%%sql 
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public';

Running query in 'postgresql://postgres:***@db'

22 rows affected.

table_name
companies
jobs
employees02
badges
comments
inventory
posthistory
postlinks
posts
tags


### List attributes/columns
The following query retrieves a list of columns from the tables in the database.

In [5]:
%%sql 
SELECT table_name, column_name, data_type, is_nullable, ordinal_position
FROM information_schema.columns
WHERE table_schema = 'public' AND table_name IN ('artists', 'released_by', 'releases', 'tracks')
AND table_name NOT LIKE 'pg_%'
ORDER BY table_name, ordinal_position;

Running query in 'postgresql://postgres:***@db'

17 rows affected.

table_name,column_name,data_type,is_nullable,ordinal_position
artists,artist_id,integer,NO,1
artists,name,character varying,YES,2
artists,realname,text,YES,3
artists,profile,text,YES,4
artists,url,text,YES,5
released_by,release_id,integer,NO,1
released_by,artist_id,integer,NO,2
releases,release_id,integer,NO,1
releases,released,date,NO,2
releases,title,text,NO,3


### Have a look at the datasets
The following simple query gives the first 5 rows of the `artists` dataset

In [6]:
%%sql
SELECT * FROM artists LIMIT 5;

Running query in 'postgresql://postgres:***@db'

5 rows affected.

artist_id,name,realname,profile,url
1,The Persuader,Jesper Dahlbäck,None,None
2,Mr. James Barth & A.D.,Cari Lekebusch & Alexi Delano,None,None
3,Josh Wink,Joshua Winkelman,"After forming [l=Ovum Recordings] as an independent label in October 1994 with former partner [a=King Britt], Josh recorded the cult classic 'Liquid Summer'. He went on to release singles for a wide variety of revered European labels ranging from Belgium's [l=R & S Records] to England's [l=XL Recordings]. In 1995, Wink became one of the first DJ-producers to translate his hard work into mainstream success when he unleashed a string of classics including 'Don't Laugh'¸ 'I'm Ready' and 'Higher State of Consciousness' that topped charts worldwide. More recently he has had massive club hits such as 'How's Your Evening So Far' and 'Superfreak' but he has also gained a lot of attention trough his remixes for [a=FC Kahuna], [a=Paul Oakenfold], [a=Ladytron], [a=Clint Mansell], [a=Sting] and [a=Depeche Mode], among others.",http://www.joshwink.com/
4,Johannes Heil,Johannes Heil,"Electronic music producer, musician and live performer, born 3 February 1978 near the town of Bad Nauheim, Germany. Founder of [l=JH] and [l=Metatron Recordings].",http://johannes-heil.com/
5,Heiko Laux,Heiko Laux,German DJ and producer based in Berlin. He is the founder of [l=Kanzleramt].,http://www.heiko-laux.com


Naturally we could write similar queries to better understand each of the other tables.

#### With what you now know about the datasets, try to answer the following questions

1. Which concepts are modelled in the dataset and how do they relate to each other? <b>Hint</b>: how do the tables connect logically?
2. Why do you think this shape (table) was chosen for the data and why not the other shapes?
3. In which normal forms are the corresponding relations? What workload assumption does keeping the relations in (high) normal form reflect?
4. How can we denormalise the data to make some queries more efficient? <b>Hint</b>: have a look at the queries in the next session of the exercises to see if adding some columns to some tables could reduce the need to `JOIN`.
5. What potential problems could result from adding redundancy?
6. Give one deletion anomaly that denormalizing `artists` into `releases` could cause.

1. The dataset contains data from discogs and are related by attributes to each other (artist id on released by on releases etc.)
2. Because we have a clear and fixed schema for each attribute, so a table is best suited (ACID), attributes come from a well-defined domain.
3. All relations are atomic, so they are for sure in 1NF. Artists, tracks and releases are for sure in 2NF since they have a singular ID primary key. No transitive dependencies, so in 3NF. Released by has no non-key attributes, so it is in 3NF. 
4. We could start by merging tables, because that would avoid having to use joins when making queries. One example is the released_by table should be merged into the releases table, although we are then losing the 3NF of that table. 
5. Insert and update anomalies: we could insert now a name with a fake or wrong artist name, and if we only update an artist name but not the entry in the releases table, we have anomalies. 
6. Deleting the last release that still carries an artist’s record could erase that artist’s name from the database even though the artist should remain. Only because the artist doesnt have any releases, does not mean that the artist doesn't exist. 

## Exercise 2: SQL warm-up
Now that we familiarised ourself with the tables and relationship, we will begin with several SQL queries to ease us back into the language.

<b>Practical tips:</b>
- You might want to begin by retrieving a few rows from each of the database tables to get a sense of what is stored. 
- When testing your queries, it is good practice to add a "LIMIT" clause to avoid inadvertedly retrieving hundreds of rows.

The following is an example query that contains some common SQL expressions. A complete list can be found at: https://www.postgresql.org/docs/current/sql-select.html

In [7]:
%%sql
SELECT DISTINCT
    a.name AS column1,
    COUNT(t.track_id) AS column2,
    AVG(t.duration) AS column3
FROM
    artists a
    JOIN released_by rb USING(artist_id)
    JOIN releases r USING(release_id)
    JOIN tracks t USING(release_id)
WHERE
    t.duration > 123
    AND t.title != 'My Query'
    AND r.country = 'Switzerland'
GROUP BY
    a.artist_id, a.name
HAVING
    COUNT(t.track_id) > 0
ORDER BY
    column2 DESC,
    column3 DESC
LIMIT 5;

Running query in 'postgresql://postgres:***@db'

5 rows affected.

column1,column2,column3
Various Artists,2814,349.5806680881307747
DJ Snowman,548,287.5620437956204380
DJ Noise,504,401.8650793650793651
Dave 202,425,326.8800000000000000
DJ Nonsdrome,242,306.3842975206611570


The following is a visual representation of the database schema for quick reference.

<img src="https://polybox.ethz.ch/index.php/s/8CqNffQrR0EDbuC/download" width=800/>

#### 1. Retrieve all releases that were released after January 1, 2017.

In [9]:
%%sql
SELECT *
FROM releases
WHERE released > '2017-01-01';

Running query in 'postgresql://postgres:***@db'

18 rows affected.

release_id,released,title,country,genre
100000,2018-12-14,"Kizomba Mix, Vol. 2 [2018] 2 CDs",Portugal,Electronic
138522,2018-01-01,Jetlag Disco,Germany,Electronic
264023,2018-01-01,The Bad Behaviour E.P,UK,Electronic
266607,2018-01-01,Volume 1,UK,Electronic
376880,2018-01-01,Technikal Support,UK,Electronic
406547,2018-01-01,Melburn / Into The Storm,UK,Electronic
482188,2018-01-01,Troika,Sweden,Electronic
618428,2018-01-01,Tribal Natty (Aphrodite Remix),UK,Electronic
648035,2018-01-01,Flowers From The Unknown,Denmark,Electronic
648051,2018-01-01,Outlines,Denmark,Electronic


#### 2. Find all tracks with a duration longer than 7 hours. Assume the 'duration' column in the 'tracks' table is in seconds.

In [10]:
%%sql
SELECT *
FROM tracks
WHERE duration > 25200;

Running query in 'postgresql://postgres:***@db'

3 rows affected.

release_id,position,title,duration,track_id
47796,9,Rapper's Relight,25579,256970
47796,11,Dialectical Transformation III Peace In Rwanda Mix,27196,256972
478281,4,Live 1996.12.30.,31934,2526159


#### 3. Retrieve the titles of 5 releases along with the names of the artists who released them.

In [11]:
%%sql
SELECT r.title, a.name
FROM releases r
JOIN released_by rb USING(release_id)
JOIN artists a USING(artist_id)
LIMIT 5;

Running query in 'postgresql://postgres:***@db'

5 rows affected.

title,name
Stockholm,The Persuader
Knockin' Boots Vol 2 Of 2,Mr. James Barth & A.D.
Profound Sounds Vol. 1,Josh Wink
Flowerhead,DATacide
Knockin' Boots (Vol 1 Of 2),Mr. James Barth & A.D.


#### 4. List each genre and the number of releases in that genre.

In [12]:
%%sql
SELECT genre, COUNT(*)
FROM releases
GROUP BY genre;

Running query in 'postgresql://postgres:***@db'

15 rows affected.

genre,count
Blues,48
Brass & Military,4
Children's,6
Classical,257
Electronic,183766
"Folk, World, & Country",101
Funk / Soul,3674
Hip Hop,10598
Jazz,3325
Latin,153


#### 5. Identify the top 5 artists who have the most releases.

In [16]:
%%sql
SELECT a.name, COUNT(DISTINCT rb.release_id) AS num_releases
FROM artists a
JOIN released_by rb USING(artist_id)
GROUP BY a.artist_id
ORDER BY num_releases DESC
LIMIT 5;

Running query in 'postgresql://postgres:***@db'

5 rows affected.

name,num_releases
Various Artists,46123
Madonna,617
Pet Shop Boys,600
Faithless,336
Michael Jackson,332


#### 6. Find the artist who has the longest total duration of tracks across all their releases.

In [17]:
%%sql
SELECT a.name, SUM(t.duration) AS total_duration
FROM artists a
JOIN released_by rb USING(artist_id)
JOIN tracks t USING(release_id)
GROUP BY a.artist_id, a.name
ORDER BY total_duration DESC
LIMIT 1;

Running query in 'postgresql://postgres:***@db'

1 rows affected.

name,total_duration
Various Artists,227180023


#### 7. Find how many releases that have tracks with duplicate titles.

In [18]:
%%sql
SELECT release_id, title FROM (
    SELECT release_id, title, COUNT(*) FROM tracks
    GROUP BY release_id, title
    HAVING COUNT(*) > 1
)
LIMIT 5;

Running query in 'postgresql://postgres:***@db'

5 rows affected.

release_id,title
110,Untitled
466,Scarlet
695,Communications From The Lab
695,Last Tribute From The 20th Century
710,Meanwhile Back At Home


#### 8. Retrieve the artists with the name of 'Coldplay'.

In [19]:
%%sql
SELECT * FROM artists
WHERE name = 'Coldplay';

Running query in 'postgresql://postgres:***@db'

1 rows affected.

artist_id,name,realname,profile,url
29735,Coldplay,None,"Coldplay is an English rock band from London, England. They've been a band since January 16, 1998 when they lost a demotape competition on XFM in London. Philip Christopher Harvey is the band's manager. [b][u]Line-up:[/u][/b] Jonny Buckland (Jonathan Mark Buckland) - Guitar Will Champion (William Champion) - Drums Guy Berryman (Guy Rupert Berryman) - Bass Chris Martin (Christopher Anthony John Martin) - Vocals",http://coldplay.com/


#### 9. List the titles of all releases by that artist in alphabetical order.
<b>Hint</b>: Ignore the fact that different relases can have the same title.

In [20]:
%%sql
SELECT title FROM  (SELECT artist_id FROM artists
WHERE name = 'Coldplay')
JOIN released_by rb USING(artist_id)
JOIN releases r USING(release_id)
ORDER BY title ASC;


Running query in 'postgresql://postgres:***@db'

70 rows affected.

title
Acoustic
Acoustic
A Rush Of Blood To The Head
A Rush Of Blood To The Head
A Rush Of Blood To The Head
Boot Of Sound
Brothers & Sisters
Clocks
Clocks
Clocks


#### 10. How many tracks from 'Coldplay' have position '1'?

In [21]:
%%sql
SELECT COUNT(*)
FROM tracks t
JOIN released_by rb USING(release_id)
JOIN artists a USING(artist_id)
WHERE a.name = 'Coldplay' AND t.position = '1';

Running query in 'postgresql://postgres:***@db'

1 rows affected.

count
32


#### 11. List the titles and durations of all tracks by 'Coldplay' whose title contains 'Remix'.

In [22]:
%%sql
SELECT title, duration
FROM tracks t
JOIN released_by rb USING(release_id)
JOIN artists a USING(artist_id)
WHERE a.name = 'Coldplay' AND t.title ILIKE '%Remix%';

Running query in 'postgresql://postgres:***@db'

20 rows affected.

title,duration
Don't Need No Trouble (Lost Witness Remix),348
God Put A Smile Upon Your Face (Def Inc. Remix),510
Clocks (The Trance Remix),461
Chime (Hard House/Trance Remix),461
Square One (Remix),198
Livin' On A Prayer (Remix),198
Trouble (Lost Witness Remix),473
Clocks (Remix),482
Speed Of Sound (Karl G Remix),511
Talk (Thin White Duke Remix),258


#### 12. List the titles of all releases by 'Coldplay' that contain less than 2 tracks.

In [23]:
%%sql
SELECT r.title FROM (SELECT artist_id FROM artists
WHERE name = 'Coldplay')
JOIN released_by rb USING(artist_id)
JOIN releases r USING(release_id)
JOIN tracks t USING(release_id)
GROUP BY r.title
HAVING COUNT(*) < 2;

Running query in 'postgresql://postgres:***@db'

9 rows affected.

title
Boot Of Sound
Clocks (Cosmos Rmx)
Clocks (Dean Coleman Remix)
Clocks (Remix)
God
One I Love
Speed Of Sound (Karl G Remix)
Time
Trouble In Paradise


#### 13. What is the average track duration?

In [24]:
%%sql
SELECT AVG(duration) FROM tracks;

Running query in 'postgresql://postgres:***@db'

1 rows affected.

avg
325.0749298696787647


#### 14. How many artists have released tracks longer than twice the average?

In [25]:
%%sql
SELECT COUNT(*)
FROM artists a
JOIN released_by rb USING(artist_id)
JOIN tracks t USING(release_id)
WHERE t.duration > (SELECT 2 * AVG(duration) FROM tracks);

Running query in 'postgresql://postgres:***@db'

1 rows affected.

count
51491


#### 15. How many titles of 'Rock' releases from the 'UK' credit more than one artist?

In [26]:
%%sql
SELECT COUNT(*) FROM
(SELECT rb.release_id FROM released_by rb
JOIN releases r USING(release_id)
WHERE r.country = 'UK' AND r.genre = 'Rock'
GROUP BY rb.release_id
HAVING COUNT(rb.artist_id) > 1);

Running query in 'postgresql://postgres:***@db'

1 rows affected.

count
38


## Exercise 3: more SQL
We will now see more complex SQL queries.

<b>Practical tips:</b>

When writing complex queries, you might want to split them into smaller parts by using <b>Common Table Expressions</b> (CTEs). A CTE is a named temporary result set that you can reference within statements (SELECT, INSERT, UPDATE, ... ). You can find more about CTEs at: https://www.postgresql.org/docs/current/queries-with.html

The following is an example of a query using two CTEs:

In [27]:
%%sql
WITH countries AS (
    SELECT DISTINCT country FROM releases
),
genres AS (
    SELECT DISTINCT genre FROM releases
)
SELECT c.country as column1, g.genre as column2
FROM countries c, genres g
LIMIT 5;

Running query in 'postgresql://postgres:***@db'

5 rows affected.

column1,column2
Argentina,Blues
Argentina,Brass & Military
Argentina,Children's
Argentina,Classical
Argentina,Electronic


In some exercises, you might also want to use <b>subqueries</b>. A subquery is a nested query, usually with the purpose of retrieving data that will be used in in the outer query. For instance, subqueries can appear in WHERE, FROM and SELECT clauses.

The following is an example of a query than includes a subquery:

In [28]:
%%sql
SELECT release_id as column1 FROM (
    SELECT release_id, title, COUNT(*) FROM tracks
    GROUP BY release_id, title
    HAVING COUNT(*) > 1
) sub
LIMIT 5;

Running query in 'postgresql://postgres:***@db'

5 rows affected.

column1
110
466
695
695
710


#### 1. What is the title of the album from 'Coldplay' with the most amount of tracks?

In [29]:
%%sql
WITH coldplay_releases AS (
    SELECT r.release_id, r.title
    FROM releases r
    JOIN released_by rb USING(release_id)
    JOIN artists a USING(artist_id)
    WHERE a.name = 'Coldplay'
)
SELECT coldplay_releases.title
FROM coldplay_releases
JOIN tracks t USING(release_id)
GROUP BY coldplay_releases.release_id, coldplay_releases.title
ORDER BY COUNT(*) DESC
LIMIT 1;

Running query in 'postgresql://postgres:***@db'

1 rows affected.

title
X&Y (Special Dutch Edition)


#### 2. What is the name of the first artist in alphabetical order with releases in the most genres. Please make sure to exclude "Various Artists".

In [73]:
%%sql
WITH artist_genre AS (
    SELECT COUNT(DISTINCT r.genre) AS num_genres, a.artist_id, a.name
    FROM artists a
    JOIN released_by rb USING(artist_id)
    JOIN releases r USING(release_id)
    GROUP BY a.artist_id, a.name
    ORDER BY num_genres DESC, a.name ASC
),
artist_genre_cleaned AS (
    SELECT name, num_genres
    FROM artist_genre
    WHERE name != 'Various Artists'
)
SELECT * FROM artist_genre_cleaned LIMIT 1;

Running query in 'postgresql://postgres:***@db'

1 rows affected.

name,num_genres
Diana Ross,7


#### 3. In what year did they (the artist from the previous question) release their first album?

In [30]:
%%sql
WITH artist_genre AS (
    SELECT COUNT(DISTINCT r.genre) AS num_genres, a.artist_id, a.name
    FROM artists a
    JOIN released_by rb USING(artist_id)
    JOIN releases r USING(release_id)
    GROUP BY a.artist_id, a.name
    ORDER BY num_genres DESC, a.name ASC
),
artist_genre_cleaned AS (
    SELECT artist_id, num_genres
    FROM artist_genre
    WHERE name != 'Various Artists'
),
artist_most_genre AS (SELECT artist_id FROM artist_genre_cleaned LIMIT 1),
earliest_release AS (
    SELECT MIN(released) AS min_released FROM releases
    JOIN released_by USING(release_id)
    WHERE artist_id IN (SELECT artist_id FROM artist_most_genre)
)
SELECT EXTRACT(YEAR FROM min_released) AS earliest_year
FROM earliest_release;


Running query in 'postgresql://postgres:***@db'

1 rows affected.

earliest_year
1967


#### 4. How many artists have released an album with total track duration above twice the average total track duration?

<b>Hint</b>: this is not the same as exercise 2.13 since we are lookong at the <b>total</b> track duration of the album.

In [41]:
%%sql
WITH total_track_duration_per_album AS (
    SELECT a.artist_id, rb.release_id, SUM(t.duration) AS total_duration
    FROM artists a
    JOIN released_by rb USING(artist_id)
    JOIN tracks t USING(release_id)
    GROUP BY a.artist_id, rb.release_id
)
SELECT COUNT(DISTINCT artist_id) FROM total_track_duration_per_album
WHERE total_duration > (SELECT 2*AVG(total_duration) FROM total_track_duration_per_album);

Running query in 'postgresql://postgres:***@db'

1 rows affected.

count
3430


#### 5. How many artists have both a release with a track longer than twice the average and one with total duration longer than twice the average?

<b>Hint</b>: you can use `INTERSECT` or `EXISTS` to write your query.

In [46]:
%%sql
WITH total_track_duration_per_album AS (
    SELECT a.artist_id, rb.release_id, SUM(t.duration) AS total_duration
    FROM artists a
    JOIN released_by rb USING(artist_id)
    JOIN tracks t USING(release_id)
    GROUP BY a.artist_id, rb.release_id
),
total_track_duration AS (
    SELECT a.artist_id, SUM(t.duration) AS total_duration
    FROM artists a
    JOIN released_by rb USING(artist_id)
    JOIN tracks t USING(release_id)
    GROUP BY a.artist_id
),
track_longer_than_avg AS (
    SELECT artist_id
    FROM total_track_duration
    WHERE total_duration > (SELECT 2*AVG(total_duration) FROM total_track_duration)
),
release_longer_than_avg AS (
    SELECT artist_id
    FROM total_track_duration_per_album
    WHERE total_duration > (SELECT 2*AVG(total_duration) FROM total_track_duration_per_album)
)
SELECT COUNT(*) FROM (
    SELECT artist_id
    FROM track_longer_than_avg
    INTERSECT
    SELECT artist_id
    FROM release_longer_than_avg
);

Running query in 'postgresql://postgres:***@db'

1 rows affected.

count
1568


#### 6. Show the artists have more than 200 releases in total but have no releases with the genre 'Pop' in reversed alphabetical order.

In [47]:
%%sql
WITH no_pop AS (
    SELECT a.artist_id, a.name
    FROM artists a
    JOIN released_by rb USING(artist_id)
    JOIN releases r USING(release_id)
    WHERE r.genre != 'Pop'
),
more_than_200 AS (
    SELECT a.artist_id, a.name
    FROM artists a
    JOIN released_by rb USING(artist_id)
    JOIN releases r USING(release_id)
    GROUP BY a.artist_id, a.name
    HAVING COUNT(r.release_id) > 200
)
SELECT artist_id, name FROM no_pop
INTERSECT
SELECT artist_id, name FROM more_than_200;

Running query in 'postgresql://postgres:***@db'

15 rows affected.

artist_id,name
48865,Kool & The Gang
7142,The Art Of Noise
194,Various Artists
1280,Underworld
10343,Tangerine Dream
7552,Pet Shop Boys
1279,Orbital
7297,The Shamen
10783,Beastie Boys
14457,Technotronic


#### 7. Excluding 'Various Artists', how many artists have at least 3 Electronic and 3 Rock releases, and whose average album duration is higher for Electronic than for Rock?

In [49]:
%%sql
WITH release_totals AS (
    SELECT release_id, SUM(duration) AS total_duration
    FROM tracks
    GROUP BY release_id
),
electronic_stats AS (
    SELECT a.artist_id, a.name, COUNT(DISTINCT r.release_id) AS num_releases, AVG(rt.total_duration) AS avg_duration
    FROM artists a
    JOIN released_by rb USING(artist_id)
    JOIN releases r USING(release_id)
    JOIN release_totals rt USING(release_id)
    WHERE r.genre = 'Electronic'
    GROUP BY a.artist_id, a.name
),
rock_stats AS (
    SELECT a.artist_id, a.name, COUNT(DISTINCT r.release_id) AS num_releases, AVG(rt.total_duration) AS avg_duration
    FROM artists a
    JOIN released_by rb USING(artist_id)
    JOIN releases r USING(release_id)
    JOIN release_totals rt USING(release_id)
    WHERE r.genre = 'Rock'
    GROUP BY a.artist_id, a.name
)
SELECT COUNT(*)
FROM electronic_stats e
JOIN rock_stats r USING(artist_id)
WHERE e.name != 'Various Artists'
  AND e.num_releases >= 3 AND r.num_releases >= 3
  AND e.avg_duration > r.avg_duration;

Running query in 'postgresql://postgres:***@db'

1 rows affected.

count
110


## Exercise 4: Discuss query patterns and language features of SQL
1. What patterns did you use in many of the queries above? 

2. What is the usual pattern of an SQL query? Which operations happen pre-grouping and which ones post-grouping?

3. What makes SQL a declarative language and what advantages does that have?

4. What makes SQL a functional language and what advantages does that have?

5. Relational algebra treats relations as sets; SQL treats tables as bags. Where did that show up in your queries, and what does `DISTINCT` / `UNION` / `INTERSECT` change?

6. How would the denormalization we talked about previously simplify the queries?

1. Subqueries, subtables. Select, joins, projections, filter, group by, order
2. Select joins and projections happen pre-grouping, filter and ordering post-grouping
3. SQL is a declarative language since you never say how to get the desired data but only what data you want. This makes the query itself implementation independent, since it doesn't matter how the DB does it in the backend. 
4. SQL is functional since queries are simply a mapping between the domains of columns. This has the advantage that running the same query again should yield the same result, no side-effects. And also since results of the query can be used as an input to a new query.
5. This shows up because you can have repeated entries inside one columnm, depending on what the primary key is, the identifier might consist of more than one attribute. DISTINCT takes the bag semantics and turns it to a set semantics. UNION and INTERSECT are set operators, so they dont return duplicates.
6. We no longer need to join the table for many queries.

## Exercise 5: Limits of SQL (optional)
Explain what the following query does.
<b>Hints</b>: The query treats the data as if it was in graph shape.

In [48]:
%%sql
WITH RECURSIVE
    X AS (SELECT 3 AS Value),
    artist_releases AS (
        SELECT artists.artist_id, artists.name, releases.release_id, releases.title
        FROM artists, released_by, releases
        WHERE artists.artist_id = released_by.artist_id
        AND released_by.release_id = releases.release_id
    ),
    collaborations AS (
        SELECT DISTINCT ar1.artist_id AS left_id, ar1.name AS left_name, 
                ar2.artist_id AS right_id, ar2.name AS right_name, 1 AS distance
        FROM artist_releases AS ar1, artist_releases AS ar2
        WHERE ar1.release_id = ar2.release_id
        AND ar1.artist_id != ar2.artist_id
    ),
    X_hop_collaborations AS (
        SELECT * FROM collaborations  -- base case
        UNION
        SELECT c1.left_id, c1.left_name, c2.right_id, c2.right_name, c1.distance + 1 AS distance
        FROM X_hop_collaborations AS c1
        JOIN collaborations c2 ON c1.right_id = c2.left_id
        WHERE c1.distance < (SELECT * FROM X) AND c1.left_id != c2.right_id
    )
SELECT * 
FROM X_hop_collaborations
WHERE left_name = 'Coldplay'
ORDER BY distance, right_name;

Running query in 'postgresql://postgres:***@db'

23 rows affected.

left_id,left_name,right_id,right_name,distance
29735,Coldplay,1654,DK,1
29735,Coldplay,392179,G Synth,1
29735,Coldplay,10916,Jan Johnston,1
29735,Coldplay,1279,Orbital,1
29735,Coldplay,10785,Angelo Badalamenti,2
29735,Coldplay,11101,Cosmic Gate,2
29735,Coldplay,7090,Freefall,2
29735,Coldplay,2604010,Jada (7),2
29735,Coldplay,18836,Kirk Hammett,2
29735,Coldplay,11749,Knuckleheadz,2


This is an implemnetation of graph traversal (Breath First Search) of the tables essentially, and it's establishing a tree of how far collaborating artsits are from Coldplay. 